# Dataset medi: cosa trovano (e cosa perdono) i tre algoritmi approssimati

<a href="https://colab.research.google.com/github/Jacop10/tesi_divexplorer/blob/main/notebooks/MediumDatasetsAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sui dataset piccoli tutti i metodi trovavano tutto. Sui dataset medi (20 attributi) i
metodi si separano, e qui guardiamo **perché**:

1. **recall nel tempo**: quanta parte dei sottogruppi importanti (T_min) ogni metodo ha
   trovato dopo t secondi;
2. **cosa perdono** beam search e MCTS: lunghezza, supporto e divergenza dei sottogruppi
   mancati rispetto a quelli trovati;
3. **con e senza pazienza**: beam search e MCTS fermati dalla pazienza oppure lasciati
   andare fino a un limite di tempo;
4. **i sottogruppi più divergenti**: chi trova i 10 con |divergenza| più alta.

Due dataset con 20 attributi:
- **sintetico**: sappiamo quali sono i gruppi nascosti, quindi possiamo controllare chi li trova;
- **ACS** (censimento USA, California 2018): dati reali, attributi interpretabili.

Ogni sezione impiega qualche minuto: beam e MCTS senza pazienza possono usare fino al
doppio del tempo di DivExplorer.

## Preparazione

In locale il notebook usa la cartella sopra `notebooks/`; su Colab scarica il repository
da GitHub e installa le librerie (comprese quelle per ACS).

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/Jacop10/tesi_divexplorer.git"
BRANCH = "jacopo"

try:
    import google.colab          # esiste solo su Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = "/content/repo"
    if not os.path.exists(os.path.join(ROOT, "divexplorer")):
        r = subprocess.run(["git", "clone", "-b", BRANCH, REPO, ROOT],
                           capture_output=True, text=True)
        print(r.stdout, r.stderr)
        if r.returncode != 0:
            raise RuntimeError("Download del repository fallito: il repository è pubblico "
                               "e il nome è giusto?")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    os.path.join(ROOT, "requirements.txt"), "folktables", "scikit-learn"])
else:
    ROOT = os.path.abspath("..")

for path in (ROOT, os.path.join(ROOT, "benchmarks")):
    sys.path.insert(0, path)
    sys.path_importer_cache.pop(path, None)
import importlib
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from divexplorer.approximate_algorithms.random_rows import random_rows
from divexplorer.approximate_algorithms.beam_search import beam_search
from divexplorer.approximate_algorithms.mcts import mcts
from divexplorer.approximate_algorithms.common import Evaluator, important_itemsets
from compare import run_divexplorer
from datasets_medium import synthetic_medium, acs

## Funzioni di supporto

- `run_all` lancia sugli stessi dati: DivExplorer, random rows, beam search e MCTS **con**
  pazienza (come in `run_medium.py`), e beam search e MCTS **senza** pazienza (si fermano
  solo al limite di tempo: il doppio del tempo di DivExplorer, almeno 30 s).
  Restituisce i risultati e T_min, l'insieme dei sottogruppi importanti secondo DivExplorer.
- `summary`: tabella riassuntiva.
- `plot_recall`: recall su T_min in funzione del tempo.
- `found_vs_missed` e `missed`: cosa trova e cosa perde ogni metodo.
- `found_by`: per una lista di sottogruppi, quali metodi li hanno trovati.

In [ ]:
COLORS = {"DivExplorer": "#2a78d6", "Random rows": "#eb6834",
          "Beam search": "#1baf7a", "MCTS": "#eda100"}


def run_all(df, attrs, outcome, kind, min_support=0.05, tau=2.0, eps=0.05,
            K=2000, K_beam=200, K_mcts=5000, seed=0):
    runs = {}
    ref, t_dx = run_divexplorer(df, attrs, outcome, kind, min_support)
    ev_ref = Evaluator(df, attrs, outcome, kind)
    T_true = important_itemsets(ref, outcome, tau, eps, ev_ref)
    runs["DivExplorer"] = {"alg": "DivExplorer", "res": ref, "ev": ev_ref,
                           "info": {"tempo": t_dx}, "T": T_true}
    print(f"DivExplorer: {t_dx:.1f} s, {len(ref)} frequenti, {len(T_true)} in T_min", flush=True)

    def add(name, alg, out):
        res, ev, info = out
        T = important_itemsets(res, outcome, tau, eps, ev) & T_true
        runs[name] = {"alg": alg, "res": res, "ev": ev, "info": info, "T": T}
        print(f"{name:30s}: {info['tempo']:.1f} s, recall {100 * len(T) / len(T_true):.1f}%  "
              f"(fermato da: {info['fermato_da']})", flush=True)

    add("Random rows", "Random rows",
        random_rows(df, attrs, outcome, kind, min_support, tau, eps, K=K, seed=seed))
    safety = 10 * runs["Random rows"]["info"]["tempo"]
    add("Beam search (pazienza)", "Beam search",
        beam_search(df, attrs, outcome, kind, min_support, tau, eps, K=K_beam,
                    time_budget=safety))
    add("MCTS (pazienza)", "MCTS",
        mcts(df, attrs, outcome, kind, min_support, tau, eps, K=K_mcts,
             time_budget=safety, seed=seed))
    limit = max(2 * t_dx, 30.0)
    add("Beam search (senza pazienza)", "Beam search",
        beam_search(df, attrs, outcome, kind, min_support, tau, eps, K=10 ** 9,
                    time_budget=limit))
    add("MCTS (senza pazienza)", "MCTS",
        mcts(df, attrs, outcome, kind, min_support, tau, eps, K=10 ** 9,
             time_budget=limit, seed=seed))
    return runs, T_true


def top_divergent(runs, outcome, k=10):
    """I k sottogruppi di T_min con |divergenza| più alta (secondo DivExplorer)."""
    ref = runs["DivExplorer"]["res"]
    imp = ref[ref["itemset"].isin(runs["DivExplorer"]["T"])]
    order = imp[f"{outcome}_div"].abs().sort_values(ascending=False).index
    return list(imp.loc[order, "itemset"][:k])


def summary(runs, T_true, outcome):
    top10 = set(top_divergent(runs, outcome))
    rows = []
    for name, r in runs.items():
        res, T = r["res"], r["T"]
        t_last = (res.loc[res["itemset"].isin(T), "t_found"].max()
                  if "t_found" in res.columns else r["info"]["tempo"])
        rows.append({"metodo": name, "tempo (s)": r["info"]["tempo"],
                     "ultimo importante (s)": t_last,
                     "frequenti": len(res), "importanti": len(T),
                     "recall T_min (%)": 100 * len(T) / len(T_true),
                     "top-10 (%)": 10 * len(T & top10),
                     "fermato da": r["info"].get("fermato_da", "fine")})
    return pd.DataFrame(rows).round(2)


def plot_recall(runs, T_true, title):
    """Recall su T_min nel tempo. Colore = algoritmo; linea continua = con pazienza,
    tratteggiata = senza pazienza. DivExplorer trova tutto solo alla fine."""
    fig, ax = plt.subplots(figsize=(9, 4))
    for name, r in runs.items():
        color = COLORS[r["alg"]]
        end = r["info"]["tempo"]
        if name == "DivExplorer":
            ax.step([0.01, end, end], [0, 0, 100], where="post", color=color, lw=2,
                    label=name)
            continue
        res = r["res"]
        times = np.sort(res.loc[res["itemset"].isin(r["T"]), "t_found"].to_numpy())
        times = np.maximum(times, 0.01)
        recall = 100 * np.arange(1, len(times) + 1) / len(T_true)
        x = np.concatenate([[0.01], times, [end]])
        y = np.concatenate([[0], recall, [recall[-1] if len(recall) else 0]])
        ax.step(x, y, where="post", color=color, lw=2,
                ls="--" if "senza" in name else "-", label=name)
    ax.set_xscale("log")
    ax.set_xlabel("tempo (s, scala logaritmica)")
    ax.set_ylabel("sottogruppi importanti trovati (% di T_min)")
    ax.set_ylim(0, 105)
    ax.set_title(title)
    ax.grid(True, color="#e6e5e0", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
    plt.tight_layout()
    plt.show()


def found_vs_missed(runs, T_true, outcome):
    """Per ogni metodo: mediana di lunghezza, supporto e |divergenza| dei sottogruppi
    importanti trovati e di quelli mancati."""
    ref = runs["DivExplorer"]["res"].set_index("itemset")
    rows = []
    for name, r in runs.items():
        if name == "DivExplorer":
            continue
        for label, group in (("trovati", r["T"]), ("mancati", T_true - r["T"])):
            sub = ref.loc[list(group)] if group else ref.iloc[:0]
            rows.append({"metodo": name, "gruppo": label, "quanti": len(group),
                         "lunghezza (mediana)": sub["length"].median(),
                         "supporto (mediana)": sub["support"].median(),
                         "|div| (mediana)": sub[f"{outcome}_div"].abs().median()})
    return pd.DataFrame(rows).round(3)


def missed(runs, name, outcome, k=10):
    """I k sottogruppi importanti mancati da un metodo, con |divergenza| più alta."""
    ref = runs["DivExplorer"]["res"]
    miss = ref[ref["itemset"].isin(runs["DivExplorer"]["T"] - runs[name]["T"])]
    cols = ["support", "itemset", outcome, f"{outcome}_div", f"{outcome}_t", "length"]
    return (miss.reindex(miss[f"{outcome}_div"].abs().sort_values(ascending=False).index)
            [cols].head(k).reset_index(drop=True))


def found_by(runs, itemsets, outcome):
    """Per ogni sottogruppo: divergenza e quali metodi lo hanno trovato come importante."""
    ref = runs["DivExplorer"]["res"]
    div = dict(zip(ref["itemset"], ref[f"{outcome}_div"]))
    rows = []
    for it in itemsets:
        row = {"itemset": ", ".join(sorted(it)), f"{outcome}_div": round(div.get(it, np.nan), 3)}
        for name, r in runs.items():
            row[name] = "✓" if it in r["T"] else "✗"
        rows.append(row)
    return pd.DataFrame(rows)

## 1. Sintetico con 20 attributi

50.000 righe, attributi `A0`…`A19` con 3 valori. Ci sono **due gruppi nascosti** in cui il
modello fa più falsi positivi: `A0=v0, A3=v0, A7=v0` e `A2=v0, A5=v0, A8=v0`.
Outcome: falsi positivi (`fp`), supporto minimo 0,05.

In [ ]:
df_syn, attrs_syn, outcome_syn, kind_syn, _ = synthetic_medium(20)
syn, T_syn = run_all(df_syn, attrs_syn, outcome_syn, kind_syn)

In [ ]:
summary(syn, T_syn, outcome_syn)

### Recall nel tempo

Ogni curva sale quando il metodo trova un nuovo sottogruppo importante. Dove la curva
diventa piatta il metodo non trova più niente: con la pazienza si ferma poco dopo.
Con e senza pazienza lo stesso algoritmo fa gli stessi passi, quindi la curva continua e
quella tratteggiata coincidono fino al punto in cui la pazienza lo ferma.

In [ ]:
plot_recall(syn, T_syn, "Sintetico, 20 attributi: recall su T_min nel tempo")

### Chi trova i gruppi nascosti

Per ogni gruppo nascosto e per i suoi sottoinsiemi di 2 condizioni: è in T_min? E quali
metodi lo hanno trovato?

In [ ]:
from itertools import combinations

hidden = [frozenset({"A0=v0", "A3=v0", "A7=v0"}), frozenset({"A2=v0", "A5=v0", "A8=v0"})]
candidates = [h for h in hidden] + [frozenset(c) for h in hidden for c in combinations(h, 2)]
table = found_by(syn, candidates, outcome_syn)
table.insert(1, "in T_min", ["sì" if c in T_syn else "no" for c in candidates])
table

### Cosa perdono beam search e MCTS

Mediana di lunghezza, supporto e |divergenza| dei sottogruppi importanti trovati e di
quelli mancati. Se i mancati sono più lunghi, il metodo non scende abbastanza in
profondità; se hanno |divergenza| più bassa, il metodo insegue solo i più divergenti.

In [ ]:
found_vs_missed(syn, T_syn, outcome_syn)

In [ ]:
missed(syn, "Beam search (pazienza)", outcome_syn)

In [ ]:
missed(syn, "MCTS (pazienza)", outcome_syn)

### I 10 sottogruppi più divergenti

In [ ]:
found_by(syn, top_divergent(syn, outcome_syn), outcome_syn)

## 2. ACS con 20 attributi

Censimento USA (California 2018, 100.000 adulti). Un random forest prevede se il reddito
supera 50.000 $; l'outcome `fp` vale 1 per i falsi positivi, 0 per i veri negativi.
Attributi: età, ore lavorate, istruzione e altre 17 colonne categoriche del censimento.
La prima volta scarica i dati e addestra il classificatore (qualche minuto), poi li riusa
da `benchmarks/data_cache/`.

In [ ]:
df_acs, attrs_acs, outcome_acs, kind_acs, _ = acs(20)
print(attrs_acs)
acs20, T_acs = run_all(df_acs, attrs_acs, outcome_acs, kind_acs)

In [ ]:
summary(acs20, T_acs, outcome_acs)

In [ ]:
plot_recall(acs20, T_acs, "ACS, 20 attributi: recall su T_min nel tempo")

### Cosa perdono beam search e MCTS

In [ ]:
found_vs_missed(acs20, T_acs, outcome_acs)

In [ ]:
missed(acs20, "Beam search (pazienza)", outcome_acs)

In [ ]:
missed(acs20, "MCTS (pazienza)", outcome_acs)

### I 10 sottogruppi più divergenti

In [ ]:
found_by(acs20, top_divergent(acs20, outcome_acs), outcome_acs)

## Cosa guardare

- **Recall nel tempo**: quale curva arriva per prima al 100%, e se arriva prima della fine
  di DivExplorer.
- **Con e senza pazienza**: le curve tratteggiate (senza pazienza) arrivano al 100%? Se sì,
  la recall bassa con pazienza dipende dallo stop; se ci arrivano solo dopo DivExplorer,
  per l'obiettivo "tutti gli importanti" non conviene usarle.
- **Trovati contro mancati**: se beam e MCTS perdono soprattutto sottogruppi lunghi o con
  |divergenza| bassa, sono metodi adatti al top-k più che a T_min.
- **Top-10 e gruppi nascosti**: anche quando la recall è bassa, i sottogruppi più
  divergenti potrebbero essere trovati.